# Module 8: Model-Based RL — Dyna-Q from Scratch

## Learning Objectives
By completing this notebook, you will:
1. Explain how Dyna-Q combines real experience and simulated (model-generated) experience
2. Implement Dyna-Q with a tabular world model on a custom GridWorld
3. Quantify sample efficiency gains from planning steps (n=5, 10, 50 vs. Q-learning)
4. Visualize model accuracy as the agent explores and refines its predictions

## Prerequisites
- Module 3: Temporal-Difference Learning (Q-learning update rule)
- Module 4: Tabular methods and state-action tables

## Estimated Time: 15 minutes

---

## The Model-Based Idea

Model-free RL (Q-learning, DQN, PPO) learns purely from real experience: every gradient update or table update requires an actual environment step. Environments can be slow (robotics, simulations) and real-world interactions expensive.

**Model-based RL** learns a **transition model** $\hat{P}(s' | s, a)$ and **reward model** $\hat{R}(s, a)$ alongside the policy/value function. Once learned, the model can be used to **plan** — simulate hypothetical transitions without touching the real environment.

**Dyna-Q** (Sutton, 1991) is the canonical combination:
1. Take a real step → update Q-table and model
2. Sample $n$ random past (s, a) pairs → simulate next states → update Q-table

With $n$ planning steps per real step, you effectively get $n+1$ Q-updates per environment interaction.

In [ ]:
# Purpose: Import dependencies
# Key Concept: Dyna-Q is tabular — no PyTorch needed, just numpy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print("Dyna-Q: Tabular model-based reinforcement learning")

## 1. Custom GridWorld with Dynamic Shortcuts

A static GridWorld is too easy to plan in. We use a world where a shortcut appears and disappears:
- **Phase 1 (steps 0-500)**: Long path is the only route
- **Phase 2 (steps 501+)**: Shortcut opens, cutting travel time

This tests whether Dyna-Q can detect and exploit environmental changes — a core challenge for model-based methods.

In [ ]:
# Purpose: Implement a GridWorld with a time-dependent shortcut
# Key Concept: The shortcut tests model staleness — the model must update when the world changes

class DynamicGridWorld:
    """
    A 6x9 GridWorld where a shortcut opens after a fixed number of steps.

    Layout (rows x cols = 6 x 9):
    - Start: bottom-left (row 5, col 0)
    - Goal:  top-right   (row 0, col 8)
    - Initial walls block the direct path
    - Shortcut wall removed after `shortcut_step` steps

    Actions: 0=up, 1=down, 2=left, 3=right
    Reward: -1 per step, +10 at goal
    """

    ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
    N_ACTIONS = 4

    def __init__(self, rows: int = 6, cols: int = 9, shortcut_step: int = 500):
        self.rows = rows
        self.cols = cols
        self.shortcut_step = shortcut_step
        self.global_step = 0

        self.start = (rows - 1, 0)
        self.goal = (0, cols - 1)

        # Walls: a vertical barrier in column 3 (rows 0-4)
        self.base_walls = set([(r, 3) for r in range(rows - 1)])
        # Shortcut: removes the wall at row 0, col 3 (top gap)
        self.shortcut_wall = (0, 3)

        self.state = self.start

    def _walls(self) -> set:
        """Return current wall set based on global step count."""
        if self.global_step < self.shortcut_step:
            return self.base_walls
        else:
            # Shortcut opens: remove the top wall segment
            return self.base_walls - {self.shortcut_wall}

    def reset(self) -> tuple:
        """Reset agent to start position."""
        self.state = self.start
        return self.state

    def step(self, action: int) -> tuple:
        """
        Take a step in the environment.

        Returns
        -------
        tuple of (next_state, reward, done)
        """
        self.global_step += 1
        dr, dc = self.ACTIONS[action]
        r, c = self.state
        nr, nc = r + dr, c + dc

        # Clip to grid bounds
        nr = max(0, min(self.rows - 1, nr))
        nc = max(0, min(self.cols - 1, nc))

        # Check wall collision — stay in place
        if (nr, nc) in self._walls():
            nr, nc = r, c

        self.state = (nr, nc)
        done = (self.state == self.goal)
        reward = 10.0 if done else -1.0

        return self.state, reward, done

    @property
    def n_states(self) -> int:
        return self.rows * self.cols

    def state_to_idx(self, state: tuple) -> int:
        return state[0] * self.cols + state[1]


# Quick test
env = DynamicGridWorld()
s = env.reset()
s2, r, done = env.step(0)  # move up
print(f"Start: {s} -> after 'up': {s2}, reward: {r}, done: {done}")
print(f"Grid: {env.rows}x{env.cols} | States: {env.n_states} | Actions: {env.N_ACTIONS}")

## 2. Tabular World Model

The model stores the **most recent** next-state and reward for each (state, action) pair — a deterministic model. This is appropriate for deterministic environments. For stochastic environments, we would track transition counts and reward distributions.

In [ ]:
# Purpose: Implement the tabular transition and reward model
# Key Concept: The model stores (s, a) -> (s', r) pairs from real experience

class TabularModel:
    """
    Deterministic tabular environment model.

    Stores the most recently observed (next_state, reward) for each (state, action).
    Used for Dyna-Q planning steps.
    """

    def __init__(self):
        # model[(s, a)] = (s', r)
        self.model = {}
        # Track all visited (s, a) pairs for uniform sampling
        self.visited = []

    def update(self, state: tuple, action: int, next_state: tuple, reward: float):
        """Add or update a transition in the model."""
        if (state, action) not in self.model:
            self.visited.append((state, action))
        self.model[(state, action)] = (next_state, reward)

    def sample(self) -> tuple:
        """
        Sample a random previously-visited (state, action) pair.

        Returns
        -------
        tuple of (state, action, next_state, reward)
        """
        idx = np.random.randint(len(self.visited))
        s, a = self.visited[idx]
        s_prime, r = self.model[(s, a)]
        return s, a, s_prime, r

    def is_ready(self) -> bool:
        """True if the model has at least one transition stored."""
        return len(self.visited) > 0

    @property
    def accuracy(self) -> float:
        """Fraction of visited (s,a) pairs (placeholder for model coverage)."""
        return len(self.visited)


print("TabularModel defined.")

## 3. Q-Learning and Dyna-Q Agents

Both agents share the same Q-table update rule. Dyna-Q adds $n$ planning steps after each real step.

In [ ]:
# Purpose: Implement Q-learning update and Dyna-Q planning
# Key Concept: Planning steps are identical to real Q-updates but use model-generated transitions

class DynaQAgent:
    """
    Dyna-Q agent with tabular Q-function and world model.

    Parameters
    ----------
    n_states : int
        Total number of states.
    n_actions : int
        Number of discrete actions.
    n_planning : int
        Planning steps per real environment step (0 = pure Q-learning).
    alpha : float
        Learning rate.
    gamma : float
        Discount factor.
    epsilon : float
        Epsilon-greedy exploration rate.
    """

    def __init__(
        self,
        n_states: int,
        n_actions: int,
        n_planning: int = 0,
        alpha: float = 0.1,
        gamma: float = 0.95,
        epsilon: float = 0.1
    ):
        self.n_actions = n_actions
        self.n_planning = n_planning
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

        # Initialize Q-table to zero
        self.Q = np.zeros((n_states, n_actions))
        self.model = TabularModel()

    def select_action(self, state_idx: int) -> int:
        """Epsilon-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.Q[state_idx]))

    def _q_update(self, s_idx: int, a: int, r: float, s_prime_idx: int):
        """
        Standard Q-learning update.

        Q(s,a) <- Q(s,a) + alpha * [r + gamma * max_a' Q(s',a') - Q(s,a)]
        """
        td_target = r + self.gamma * np.max(self.Q[s_prime_idx])
        td_error = td_target - self.Q[s_idx, a]
        self.Q[s_idx, a] += self.alpha * td_error

    def update(
        self,
        state: tuple,
        action: int,
        reward: float,
        next_state: tuple,
        env_state_to_idx
    ):
        """
        Real update: Q-update + model update + n_planning simulated Q-updates.

        Parameters
        ----------
        state, next_state : tuple
            Grid positions.
        action : int
            Action taken.
        reward : float
            Observed reward.
        env_state_to_idx : callable
            Maps (row, col) to flat index.
        """
        s_idx = env_state_to_idx(state)
        sp_idx = env_state_to_idx(next_state)

        # Step 1: Real Q-update
        self._q_update(s_idx, action, reward, sp_idx)

        # Step 2: Update model
        self.model.update(state, action, next_state, reward)

        # Step 3: n_planning simulated Q-updates from model
        if self.model.is_ready():
            for _ in range(self.n_planning):
                sim_s, sim_a, sim_sp, sim_r = self.model.sample()
                sim_s_idx = env_state_to_idx(sim_s)
                sim_sp_idx = env_state_to_idx(sim_sp)
                self._q_update(sim_s_idx, sim_a, sim_r, sim_sp_idx)


print("DynaQAgent defined.")

## 4. Training Loop and Comparison

We compare:
- **Plain Q-learning** (n_planning=0)
- **Dyna-Q n=5**
- **Dyna-Q n=10**
- **Dyna-Q n=50**

All trained for 1500 real environment steps. We track cumulative reward and model size.

In [ ]:
# Purpose: Run training loop and collect performance data
# Key Concept: Cumulative reward vs. real steps shows sample efficiency

def run_experiment(
    n_planning: int = 0,
    total_steps: int = 1500,
    shortcut_step: int = 750,
    alpha: float = 0.1,
    gamma: float = 0.95,
    epsilon: float = 0.1,
    seed: int = SEED
) -> dict:
    """
    Run Dyna-Q (or Q-learning) on DynamicGridWorld.

    Returns
    -------
    dict with keys: 'steps_per_episode', 'cumulative_reward', 'model_size'
    """
    np.random.seed(seed)
    env = DynamicGridWorld(shortcut_step=shortcut_step)
    agent = DynaQAgent(
        n_states=env.n_states,
        n_actions=env.N_ACTIONS,
        n_planning=n_planning,
        alpha=alpha,
        gamma=gamma,
        epsilon=epsilon
    )

    steps_per_episode = []
    cumulative_reward = []
    model_size = []         # Number of (s,a) pairs in model

    real_step = 0
    cum_r = 0.0

    while real_step < total_steps:
        state = env.reset()
        ep_steps = 0
        done = False

        while not done and real_step < total_steps:
            s_idx = env.state_to_idx(state)
            action = agent.select_action(s_idx)
            next_state, reward, done = env.step(action)

            agent.update(state, action, reward, next_state, env.state_to_idx)

            cum_r += reward
            real_step += 1
            ep_steps += 1

            cumulative_reward.append(cum_r)
            model_size.append(agent.model.accuracy)

            state = next_state

        steps_per_episode.append(ep_steps)

    return {
        'steps_per_episode': steps_per_episode,
        'cumulative_reward': cumulative_reward,
        'model_size': model_size
    }


configs = {
    'Q-learning (n=0)': 0,
    'Dyna-Q (n=5)': 5,
    'Dyna-Q (n=10)': 10,
    'Dyna-Q (n=50)': 50
}

results = {}
for name, n_plan in configs.items():
    print(f"Running {name}...")
    results[name] = run_experiment(n_planning=n_plan)

print("All experiments complete.")

## 5. Visualizing Sample Efficiency and Model Accuracy

In [ ]:
# Purpose: Visualize cumulative reward, episode length, and model growth
# Key Concept: Steeper cumulative reward curve = better sample efficiency

COLORS = {
    'Q-learning (n=0)': 'steelblue',
    'Dyna-Q (n=5)': 'darkorange',
    'Dyna-Q (n=10)': 'crimson',
    'Dyna-Q (n=50)': 'darkgreen'
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Plot 1: Cumulative reward vs. real steps ---
ax = axes[0]
for name, data in results.items():
    ax.plot(data['cumulative_reward'], label=name, color=COLORS[name], linewidth=2)
ax.axvline(x=750, color='gray', linestyle='--', alpha=0.7, label='Shortcut opens')
ax.set_xlabel('Real Environment Steps', fontsize=12)
ax.set_ylabel('Cumulative Reward', fontsize=12)
ax.set_title('Sample Efficiency: Cumulative Reward', fontsize=13)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 2: Steps per episode (lower = policy found shorter path) ---
ax = axes[1]
for name, data in results.items():
    eps = data['steps_per_episode']
    # Smooth with moving average
    if len(eps) > 5:
        smoothed = np.convolve(eps, np.ones(5) / 5, mode='valid')
        ax.plot(smoothed, label=name, color=COLORS[name], linewidth=2)
ax.axvline(x=len(results['Q-learning (n=0)']['steps_per_episode']) // 2,
           color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Steps per Episode (5-ep avg)', fontsize=12)
ax.set_title('Policy Quality: Episode Length', fontsize=13)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Plot 3: Model size (unique s,a pairs seen) ---
ax = axes[2]
for name, data in results.items():
    ax.plot(data['model_size'], label=name, color=COLORS[name], linewidth=2)
ax.axvline(x=750, color='gray', linestyle='--', alpha=0.7, label='Shortcut opens')
ax.set_xlabel('Real Environment Steps', fontsize=12)
ax.set_ylabel('Unique (s,a) Pairs in Model', fontsize=12)
ax.set_title('Model Coverage Over Time', fontsize=13)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dyna_q_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to dyna_q_comparison.png")

## 6. Visualizing the Learned Q-Values

In [ ]:
# Purpose: Visualize the Q-table as a heatmap over the grid
# Key Concept: max Q(s, a) shows which states the agent values — high near goal

def get_trained_agent(n_planning: int, total_steps: int = 1500, seed: int = SEED):
    """Return a trained agent and environment for visualization."""
    np.random.seed(seed)
    env = DynamicGridWorld(shortcut_step=750)
    agent = DynaQAgent(
        n_states=env.n_states, n_actions=env.N_ACTIONS,
        n_planning=n_planning, alpha=0.1, gamma=0.95, epsilon=0.1
    )
    real_step = 0
    while real_step < total_steps:
        state = env.reset()
        done = False
        while not done and real_step < total_steps:
            s_idx = env.state_to_idx(state)
            action = agent.select_action(s_idx)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, env.state_to_idx)
            state = next_state
            real_step += 1
    return agent, env


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, n_plan, title in [
    (axes[0], 0, 'Q-learning (n=0): max Q(s,a)'),
    (axes[1], 50, 'Dyna-Q (n=50): max Q(s,a)')
]:
    agent, env = get_trained_agent(n_plan)

    # Reshape Q-table max values into grid
    q_max = np.max(agent.Q, axis=1).reshape(env.rows, env.cols)

    im = ax.imshow(q_max, cmap='RdYlGn', aspect='auto')
    plt.colorbar(im, ax=ax, label='max Q(s,a)')

    # Mark walls
    for (r, c) in env.base_walls:
        ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                        facecolor='black', alpha=0.8))

    # Mark start and goal
    ax.scatter([env.start[1]], [env.start[0]], s=200, c='blue',
               zorder=5, label='Start', marker='o')
    ax.scatter([env.goal[1]], [env.goal[0]], s=200, c='gold',
               zorder=5, label='Goal', marker='*')

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('dyna_q_values.png', dpi=100, bbox_inches='tight')
plt.show()
print("Q-value heatmaps saved to dyna_q_values.png")

## Exercise 8.1: Stochastic Model

**Task:** The current `TabularModel` stores only the most recent transition — it's deterministic. Extend it to track **transition counts** and return the most frequent next state when sampled. This makes the model suitable for stochastic environments.

Implement `StochasticTabularModel` with the same interface (`update`, `sample`, `is_ready`) but tracking counts internally.

<details>
<summary>Hint 1</summary>
Use a nested dict: `self.counts[(s, a)][s'] = count` to track how often each next state is observed.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
In `sample()`, once you've chosen a (s, a) pair, find `s' = argmax counts[(s,a)]` and use the stored mean reward for that (s, a, s') triple.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

class StochasticTabularModel:
    """
    Tabular model that tracks transition frequencies for stochastic environments.

    Stores count of each (s, a) -> s' transition and mean reward.
    During planning, samples the most frequent next state.
    """

    def __init__(self):
        # YOUR CODE HERE: initialize data structures
        pass

    def update(self, state: tuple, action: int, next_state: tuple, reward: float):
        """Update transition counts and reward estimates."""
        # YOUR CODE HERE
        pass

    def sample(self) -> tuple:
        """
        Sample a random (s, a) and return its most frequent next state.

        Returns
        -------
        tuple of (state, action, next_state, reward)
        """
        # YOUR CODE HERE
        pass

    def is_ready(self) -> bool:
        """True if at least one transition has been stored."""
        # YOUR CODE HERE
        pass

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_8_1():
    m = StochasticTabularModel()
    assert m is not None, "StochasticTabularModel must be defined"
    assert not m.is_ready(), "Empty model should not be ready"

    # Add one transition
    m.update((0,0), 0, (0,1), -1.0)
    assert m.is_ready(), "Model with one transition should be ready"

    # Sample should return the stored transition
    s, a, sp, r = m.sample()
    assert (s, a) == ((0,0), 0), f"Expected state (0,0), action 0, got ({s}, {a})"
    assert sp == (0, 1), f"Expected next_state (0,1), got {sp}"
    assert abs(r - (-1.0)) < 1e-6, f"Expected reward -1.0, got {r}"

    # Most frequent next state should win
    m2 = StochasticTabularModel()
    m2.update((1,1), 1, (1,2), -1.0)
    m2.update((1,1), 1, (1,2), -1.0)
    m2.update((1,1), 1, (1,3), -1.0)  # minority
    _, _, sp2, _ = m2.sample()
    assert sp2 == (1, 2), f"Most frequent next state should be (1,2), got {sp2}"

    print("Exercise 8.1 passed! Stochastic tabular model correct.")

test_exercise_8_1()

## Exercise 8.2: Prioritized Sweeping

**Task:** Standard Dyna-Q samples planning states uniformly at random. **Prioritized sweeping** instead samples transitions with the highest expected Q-value change — the states where the model update will matter most.

Implement `get_priority(state, action, Q, env_state_to_idx, gamma)` which computes the priority for a transition `(s, a) -> (s', r)` stored in a model. Priority = absolute TD error:

$$|r + \gamma \max_{a'} Q(s', a') - Q(s, a)|$$

<details>
<summary>Hint 1</summary>
The function needs to look up the stored (s', r) from the model. Pass the model as an argument.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def get_priority(
    state: tuple,
    action: int,
    model: TabularModel,
    Q: np.ndarray,
    env_state_to_idx,
    gamma: float = 0.95
) -> float:
    """
    Compute absolute TD error for a (state, action) pair stored in the model.

    Parameters
    ----------
    state : tuple
        Current grid position.
    action : int
        Action taken.
    model : TabularModel
        Contains (state, action) -> (next_state, reward).
    Q : np.ndarray
        Current Q-table, shape (n_states, n_actions).
    env_state_to_idx : callable
        Maps (row, col) to flat index.
    gamma : float
        Discount factor.

    Returns
    -------
    float
        Absolute TD error (priority).
    """
    your_answer = None  # Replace with your implementation
    return your_answer

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_8_2():
    env_t = DynamicGridWorld()
    model_t = TabularModel()
    Q_t = np.zeros((env_t.n_states, env_t.N_ACTIONS))

    # Store a transition: (0,0) --action 3--> (0,1), reward -1
    model_t.update((0, 0), 3, (0, 1), -1.0)

    # With Q=0 everywhere: TD target = -1 + 0.95*0 = -1; Q(s,a)=0; error = |-1-0| = 1.0
    priority = get_priority((0, 0), 3, model_t, Q_t, env_t.state_to_idx)
    assert priority is not None, "get_priority must return a float, not None"
    assert abs(priority - 1.0) < 0.01, \
        f"Expected priority ~1.0, got {priority}"

    # After Q is updated: Q(s,a) = -1 -> error = 0
    s_idx = env_t.state_to_idx((0, 0))
    Q_t[s_idx, 3] = -1.0
    priority2 = get_priority((0, 0), 3, model_t, Q_t, env_t.state_to_idx)
    assert abs(priority2) < 0.01, \
        f"After perfect Q update, priority should be ~0, got {priority2}"

    print("Exercise 8.2 passed! Priority computation correct.")

test_exercise_8_2()

## Key Takeaways

1. **Dyna-Q multiplies sample efficiency** by $n+1$ per real step — every environment interaction triggers $n$ additional model-based Q-updates at negligible compute cost.

2. **The model is a bottleneck**: planning is only as good as model accuracy. Errors in the model propagate to Q-values via simulated transitions.

3. **Dynamic environments expose model staleness**: when the shortcut opens, the model still predicts the old wall. Agents with more planning steps may be *slower* to adapt because they over-trust the stale model.

4. **Prioritized sweeping** dramatically improves planning efficiency by focusing updates on states where Q-values change the most, rather than sampling uniformly.

5. **Tabular vs. neural models**: tabular Dyna-Q scales to small discrete MDPs. For large state spaces, the model becomes a neural network — the core of Dreamer, MBPO, and World Models.

## What's Next

`02_mcts_tic_tac_toe.ipynb` explores a different form of model-based planning: **Monte Carlo Tree Search**. Rather than learning an approximate model from data, MCTS uses a perfect simulator and searches the game tree to find good actions.

## Additional Resources

- Sutton & Barto, Chapter 8: Planning and Learning with Tabular Methods
- Sutton (1991): "Dyna, an Integrated Architecture for Learning, Planning, and Reacting"
- Ha & Schmidhuber (2018): "World Models" — neural Dyna-Q at scale